In [ ]:
from playwright.async_api import async_playwright
import json
url = "https://cellphones.com.vn/mobile.html"


async def close_if_appears(page, selector, timeout=3000):
    try:
        await page.wait_for_selector(selector, state="visible", timeout=timeout)
        await page.wait_for_selector(selector, state="stable")
        await page.wait_for_timeout(1000)
        print(f"Closed popup: {selector}")

    except:
        pass



all_links = []

async with async_playwright() as p:
    browser = await p.chromium.launch_persistent_context(
        headless=False,
        slow_mo=1000,
        args=[
            "--disable-gpu",
            "--disable-dev-shm-usage",
            "--disable-extensions",
            "--disable-background-networking",
            "--disable-sync",
            "--disable-notifications",
            "--disable-default-apps",
            "--no-sandbox",
        ],
        user_data_dir="profile",
    )

    page = await browser.new_page()

    await page.route("**/*", lambda route: (
        route.abort()
        if route.request.resource_type in ["image", "font", "media"]
        else route.continue_()
    ))

    await page.goto(url)
    await page.wait_for_load_state("domcontentloaded")

    # await close_if_appears(page, "button.modal-close.is-large", 10000)
    # await close_if_appears(page, "button.cancel-button-top", 30000)

    menu_button = page.locator(".navbar__item.button__menu")
    await menu_button.click()

    do_gia_dung = page.locator("div.label-menu-tree", has_text="Đồ gia dụng")
    await do_gia_dung.hover()

    menu_titles = [
        "Gia dụng nhà bếp",
        "Sức khỏe - Làm đẹp",
        "Thiết bị gia đình",
    ]



    for title in menu_titles:
        print(f"Waiting for menu: {title}")

        block = page.locator(
            "div.menu-child-item",
            has=page.locator("strong", has_text=title),
        )

        await block.wait_for(state="visible", timeout=15000)

        links = await block.locator("a.label-wrapper").all()

        for a in links:
            href = await a.get_attribute("href")
            all_links.append(href)
    with open("all_links.json", "w", encoding="utf-8") as f:
        json.dump(all_links, f, ensure_ascii=False, indent=2)
    await browser.close()


Waiting for menu: Gia dụng nhà bếp
Waiting for menu: Sức khỏe - Làm đẹp
Waiting for menu: Thiết bị gia đình


In [4]:
all_category_links = []
all_links = []
with open("all_links.json", "r", encoding="utf-8") as f:
    links = json.load(f)
    for link in links:
        all_links.append(link)
print(all_links[:5])
async with async_playwright() as p:
    browser = await p.chromium.launch_persistent_context(
        headless=False,
        slow_mo=1000,
        args=[
            "--disable-gpu",
            "--disable-dev-shm-usage",
            "--disable-extensions",
            "--disable-background-networking",
            "--disable-sync",
            "--disable-notifications",
            "--disable-default-apps",
            "--no-sandbox",
        ],
        user_data_dir="profile",
    )
    page = await browser.new_page()

    await page.route("**/*", lambda route: (
        route.abort()
        if route.request.resource_type in ["image", "font", "media"]
        else route.continue_()
    ))
    for url in all_links:
        await page.goto(url)

        items = page.locator("a.product__link.button__link")

        show_more_button = page.locator(
            "a.button.btn-show-more.button__show-more-product"
        )
        try:
            await show_more_button.wait_for(state="visible", timeout=2000)
            for i in range(100):
                await show_more_button.click()
                if await show_more_button.is_hidden():
                    break
            # await page.wait_for_function(
            #     expression="(prev) => document.querySelectorAll('a.product__link.button__link').length > prev",
            #     arg=before_count)
            # after_count = await items.count()
            # print(f"After click: {after_count}")
        except:
            print("Unnecessary to click show more button")

        all_items_links = []
        for i in range(await items.count()):
            item = items.nth(i)
            link = await item.get_attribute("href")
            all_items_links.append(link)
        print(all_items_links)
        all_category_links.extend(all_items_links)
    await browser.close()

['https://cellphones.com.vn/do-gia-dung/noi-chien-khong-dau.html', 'https://cellphones.com.vn/do-gia-dung/lo-vi-song.html', 'https://cellphones.com.vn/do-gia-dung/noi-com-dien.html', 'https://cellphones.com.vn/do-gia-dung/may-xay-sinh-to.html', 'https://cellphones.com.vn/do-gia-dung/may-ep-trai-cay.html']
['https://cellphones.com.vn/noi-chien-khong-dau-gaabor-af-45t01a-5l.html', 'https://cellphones.com.vn/noi-chien-khong-dau-fujihome-a8dg2-8l.html', 'https://cellphones.com.vn/noi-chien-khong-dau-gaabor-6-5-lit-af65t-bk01a.html', 'https://cellphones.com.vn/noi-chien-khong-dau-gaabor-af-80t01a-6-2-lit.html', 'https://cellphones.com.vn/noi-chien-khong-dau-philips-na342-00.html', 'https://cellphones.com.vn/noi-chien-khong-dau-xiaomi-smart-air-fryer-6-5-lit.html', 'https://cellphones.com.vn/noi-chien-khong-dau-sunhouse-9l-shd4037.html', 'https://cellphones.com.vn/lo-chien-khong-dau-toshiba-tl2-sac25gzc-rd.html', 'https://cellphones.com.vn/noi-chien-khong-dau-gaabor-af-80m01a-6-2-lit.html', 

TargetClosedError: Page.goto: Target page, context or browser has been closed
Call log:
  - navigating to "https://cellphones.com.vn/do-gia-dung/lo-vi-song.html", waiting until "load"


In [8]:
import json
print(len(all_category_links))
with open("all_category_links.json", "w",  encoding="utf-8") as file:
  json.dump(all_category_links, file, ensure_ascii=False, indent=2)

938


In [5]:
import re
from playwright.async_api import async_playwright
import json

url_current_count = 0

def save_item(item):
    with open("data.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

async with async_playwright() as p:
    browser = await p.chromium.launch(
        headless=False,
        slow_mo=1000,
        args=[
            "--disable-gpu",
            "--disable-dev-shm-usage",
            "--disable-extensions",
            "--disable-background-networking",
            "--disable-sync",
            "--disable-notifications",
            "--disable-default-apps",
            "--no-sandbox",
        ],
    )
    context = await browser.new_context(
        # viewport={"width": 1280, "height": 800},
        # java_script_enabled=True,
    )

    page = await browser.new_page()
    await page.route("**/*", lambda route, request:
    route.abort()
    if request.resource_type in ["image", "font", "media"]
    else route.continue_()
)

    page.set_default_timeout(10000)
    all_item_details = []
    all_category_links = []
    with open("all_category_links.json", "r", encoding="utf-8") as f:
        links = json.load(f)
        for link in links:
            all_category_links.append(link)
    
    for url in all_category_links[499 + 241 + 38 + 17:]:
        await page.goto(url,  wait_until="domcontentloaded", timeout=30000)
        url_current_count += 1
        print(f"Processing URL {url_current_count}: {url}")

        crawl_results = {
            "url": url,
            "name": None,
            "base_price": None,
            "sale_price": None,
            "version": [],
            "colors": {},
            "images": [],
            "specs": {},
            "warranty": [],
            "promotions": [],
            "description": None,
            "comments": [],
        }

        # ---------- BASIC INFO ----------
        if await page.locator(".box-product-name > h1").count() > 0:
            crawl_results["name"] = await page.locator(
                ".box-product-name > h1"
            ).inner_text()

        if await page.locator(".box-product-price del.base-price").count() > 0:
            crawl_results["base_price"] = await page.locator(
                ".box-product-price del.base-price"
            ).inner_text()

        if await page.locator(".box-product-price .sale-price").count() > 0:
            crawl_results["sale_price"] = await page.locator(
                ".box-product-price .sale-price"
            ).inner_text()

        # ---------- VERSION ----------
        version_options = page.locator(".list-linked a.item-linked.button__link")
        for i in range(await version_options.count()):
            strong = version_options.nth(i).locator("strong")
            text = await strong.text_content()
            if text:
                crawl_results["version"].append(text.strip())


        # ---------- COLORS ----------
        color_options = page.locator("li.item-variant")
        for i in range(await color_options.count()):
            option = color_options.nth(i)

            color = None
            price = None

            if await option.locator("strong.item-variant-name").count() > 0:
                color = await option.locator(
                    "strong.item-variant-name"
                ).inner_text()

            if await option.locator("span.item-variant-price").count() > 0:
                price = await option.locator(
                    "span.item-variant-price"
                ).inner_text()

            if color:
                crawl_results["colors"][color] = price

        # ---------- IMAGES ----------
        gallery_images = page.locator(".box-gallery a.spotlight img")
        for i in range(await gallery_images.count()):
            src = await gallery_images.nth(i).get_attribute("src", timeout=30000)
            if src:
                crawl_results["images"].append(src)

        # ---------- SPECS ----------
        specs_rows = page.locator("tr.technical-content-item")
        for i in range(await specs_rows.count()):
            row = specs_rows.nth(i)
            tds = row.locator("td")

            if await tds.count() >= 2:
                key = await tds.nth(0).inner_text()
                value_locator = tds.nth(1).locator("p")

                value = (
                    await value_locator.inner_text()
                    if await value_locator.count() > 0
                    else None
                )

                crawl_results["specs"][key] = value

        # ---------- WARRANTY ----------
        warranty_rows = page.locator(".item-warranty-info")
        for i in range(await warranty_rows.count()):
            crawl_results["warranty"].append(
                await warranty_rows.nth(i).inner_html()
            )

        # ---------- PROMOTIONS ----------
        banners = page.locator(".block-special-promotion-banner a")
        for i in range(await banners.count()):
            banner = banners.nth(i)

            href = await banner.get_attribute("href")
            img = banner.locator("img")

            crawl_results["promotions"].append({
                "url": href if href and href.startswith("http") else f"https://www.cellphones.com.vn{href or ''}",
                "img": await img.get_attribute("src") if await img.count() > 0 else None,
                "alt": await img.get_attribute("alt") if await img.count() > 0 else None,
            })

        # ---------- DESCRIPTION ----------
        if await page.locator("#cpsContentSEO").count() > 0:
            crawl_results["description"] = await page.locator(
                "#cpsContentSEO"
            ).inner_html()

        # ---------- COMMENTS ----------
        comment_rows = page.locator(".item-comment__box-cmt")
        for i in range(await comment_rows.count()):
            commentSection = comment_rows.nth(i)
            comment = {"name": None, "content": None, "reply": []}

            name_locator = commentSection.locator(
                ">.box-cmt__box-info>.box-info>p.box-info__name"
            )
            if await name_locator.count() > 0:
                comment["name"] = await name_locator.inner_text()

            content_locators = commentSection.locator(
                ".box-cmt__box-question .content p"
            )
            for k in range(await content_locators.count()):
                content_locator = content_locators.nth(k)
                if await content_locator.count() > 0:
                    comment["content"] = await content_locator.inner_text()

            reply_rows = commentSection.locator(
                ">.item-comment__box-rep-comment .item-rep-comment"
            )
            for j in range(await reply_rows.count()):
                replySection = reply_rows.nth(j)
                reply = {"name": None, "content": None}

                name_locator = replySection.locator(
                    ">.box-cmt__box-info p.box-info__name"
                )
                if await name_locator.count() > 0:
                    reply["name"] = await name_locator.inner_text()

                texts = await replySection.locator(
                    ">.box-cmt__box-question .content"
                ).all_inner_texts()

                reply["content"] = "\n".join(
                    t.strip()
                    for t in texts
                    if re.search(r"[a-zA-Zà-ỹÀ-ỸđĐ]", t)
                )

                comment["reply"].append(reply)

            crawl_results["comments"].append(comment)

        save_item(crawl_results)
    await browser.close()


AttributeError: 'dict' object has no attribute '_object'

In [9]:
print(len(all_item_details))
with open('all_item_details.json', 'w', encoding="utf-8") as file:
  json.dump(all_item_details, file, ensure_ascii=False, indent=2)

498


In [ ]:
import json
from playwright.async_api import async_playwright
async with async_playwright() as p:
    browser = await p.chromium.launch_persistent_context(
        headless=False,
        slow_mo=300,
        user_data_dir="profile",
        args=[
            "--disable-gpu",
            "--disable-dev-shm-usage",
            "--disable-extensions",
            "--disable-background-networking",
            "--disable-sync",
            "--disable-notifications",
            "--disable-default-apps",
            "--no-sandbox",
        ],
    )
    page = await browser.new_page()

    await page.route("**/*", lambda route: (
        route.abort()
        if route.request.resource_type in ["image", "font", "media"]
        else route.continue_()
    ))
    page.set_default_timeout(5000)
    url = "https://cellphones.com.vn/sforum/tag/gia-dung"
    await page.goto(url)
    all_article_links = set()
    filtered_article_links = set()
    show_more_buttons = page.locator("button", has_text="Xem thêm")
    while await show_more_buttons.count() > 0:
        await show_more_buttons.first.click()
        await page.wait_for_timeout(2000)
    all_sections = page.locator("section")
    article_section = all_sections.filter(has_text='Bài viết về "Gia dụng"')
    
    a_tags = article_section.locator("a")
    for i in range(await a_tags.count()):
        link = await a_tags.nth(i).get_attribute("href")
        all_article_links.add(f"https://cellphones.com.vn{link}")
        if re.match(r"^/sforum/[a-z0-9\-]+$", link or ""):
            filtered_article_links.add(f"https://cellphones.com.vn{link}")
    print(len(filtered_article_links))
    await browser.close()

NameError: name 'async_playwright' is not defined

In [25]:
#!/usr/bin/env python3
import asyncio
import json
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

from logging.handlers import RotatingFileHandler

LOG_FILE = "crawler.log"

logger = logging.getLogger()
logger.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# ---- Console ----
console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

# ---- File (auto rotate) ----
file_handler = RotatingFileHandler(
    LOG_FILE,
    maxBytes=5 * 1024 * 1024,  # 5MB
    backupCount=5,
    encoding="utf-8"
)
file_handler.setFormatter(formatter)

# ---- Attach ----
logger.handlers.clear()
logger.addHandler(console_handler)
logger.addHandler(file_handler)


from playwright.async_api import (
    async_playwright,
    TimeoutError as PlaywrightTimeoutError,
)

HEADLESS = False
SLOW_MO = 0
NAV_RETRIES = 2
GOTO_TIMEOUT = 30_000
DEFAULT_TIMEOUT = 10_000
RESET_PAGE_BATCH = 50
BLOCK_RESOURCE_TYPES = {"image", "media"}
ALL_LINKS_FILE = "all_category_links.json"
OUTPUT_JSONL = "test.jsonl"
LOG_LEVEL = logging.INFO
CRAWL_ERROR_LOGS_FILE = "main_errors.jsonl"

logging.basicConfig(
    format="%(asctime)s %(levelname)s %(message)s",
    level=LOG_LEVEL,
    datefmt="%H:%M:%S",
)

CURRENT_ERRORS: Optional[Dict[str, Any]] = None


def append_error_log_record(record: Dict[str, Any]) -> None:
    try:
        with open(CRAWL_ERROR_LOGS_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    except Exception:
        logging.exception("Failed to write error record")


def log_error(field_name: str, source: str, err: Exception) -> None:
    global CURRENT_ERRORS
    msg = f"{source}: {type(err).__name__}: {str(err)}"
    if CURRENT_ERRORS is None:
        append_error_log_record({"url": None, "errors": {field_name: [msg]}})
        return
    errors = CURRENT_ERRORS.setdefault("errors", {})
    errors.setdefault(field_name, []).append(msg)


def save_item(item: Dict[str, Any], filename: str = OUTPUT_JSONL) -> None:
    try:
        with open(filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    except Exception:
        logging.exception("Failed to save item")


async def block_resources(route, request):
    try:
        if request.resource_type in BLOCK_RESOURCE_TYPES:
            await route.abort()
        else:
            await route.continue_()
    except Exception:
        try:
            await route.continue_()
        except Exception:
            pass


async def get_text(locator, *, field_name: str, strip: bool = True) -> Optional[str]:
    try:
        txt = await locator.first.text_content()
        if txt is None:
            return None
        return txt.strip() if strip else txt
    except Exception as e:
        log_error(field_name, "get_text", e)
        return None


async def get_all_texts(locator, field_name: str) -> List[str]:
    try:
        texts = await locator.all_inner_texts()
        return [t.strip() for t in texts if t and t.strip()]
    except Exception as e:
        log_error(field_name, "get_all_texts", e)
        return []


async def get_attribute(locator, name: str, field_name: str) -> Optional[str]:
    try:
        val = await locator.first.get_attribute(name)
        return val.strip() if isinstance(val, str) else val
    except Exception as e:
        log_error(field_name, "get_attribute", e)
        return None


async def get_inner_html(locator, field_name: str) -> Optional[str]:
    try:
        html = await locator.first.inner_html()
        return html.strip() if isinstance(html, str) else html
    except Exception as e:
        log_error(field_name, "get_inner_html", e)
        return None


async def count(locator, field_name: str) -> int:
    try:
        return await locator.count()
    except Exception as e:
        log_error(field_name, "count", e)
        return 0


async def extract_item_from_page(page, url: str) -> Dict[str, Any]:
    res: Dict[str, Any] = {
        "url": url,
        "name": None,
        "base_price": None,
        "sale_price": None,
        "version": [],
        "colors": {},
        "images": [],
        "specs": {},
        "warranty": [],
        "promotions": [],
        "description": None,
        "comments": [],
        "_meta": {"scraped_at": int(time.time())},
    }

    name = await get_text(page.locator(".box-product-name > h1"), field_name="name")
    if name:
        res["name"] = name

    base_price = await get_text(
        page.locator(".box-product-price del.base-price"), field_name="base_price"
    )
    if base_price:
        res["base_price"] = base_price

    sale_price = await get_text(
        page.locator(".box-product-price .sale-price"), field_name="sale_price"
    )
    if sale_price:
        res["sale_price"] = sale_price

    res["version"] = await get_all_texts(
        page.locator(".list-linked a.item-linked strong"), field_name="versions"
    )

    color_nodes = page.locator("li.item-variant")
    count_colors = await count(color_nodes, "colors")
    for i in range(count_colors):
        opt = color_nodes.nth(i)
        color = await get_text(
            opt.locator("strong.item-variant-name"), field_name=f"colors[{i}].name"
        )
        price = await get_text(
            opt.locator("span.item-variant-price"), field_name=f"colors[{i}].price"
        )
        if color:
            res["colors"][color] = price

    gallery = page.locator(".box-gallery a.spotlight img")
    gallery_len = await count(gallery, "images")
    for i in range(gallery_len):
        img = gallery.nth(i)
        src = await get_attribute(img, "src", f"images[{i}].src")
        if not src:
            src = await get_attribute(
                img, "data-src", f"images[{i}].data-src"
            ) or await get_attribute(img, "data-lazy", f"images[{i}].data-lazy")
        if src:
            res["images"].append(src)

    spec_rows = page.locator("tr.technical-content-item")
    spec_count = await count(spec_rows, "specs")
    for i in range(spec_count):
        row = spec_rows.nth(i)
        tds = row.locator("td")
        td_count = await count(tds, f"specs[{i}].tds")
        if td_count >= 2:
            key = await get_text(tds.nth(0), field_name=f"specs[{i}].key")
            value = None
            if await count(tds.nth(1).locator("p"), f"specs[{i}].p") > 0:
                value = await get_text(
                    tds.nth(1).locator("p"), field_name=f"specs[{i}].value"
                )
            else:
                value = await get_text(tds.nth(1), field_name=f"specs[{i}].value")
            if key:
                res["specs"][key] = value

    warranty_nodes = page.locator(".item-warranty-info")
    w_count = await count(warranty_nodes, "warranty")
    for i in range(w_count):
        h = await get_inner_html(warranty_nodes.nth(i), field_name=f"warranty[{i}]")
        if h:
            res["warranty"].append(h)

    banner_nodes = page.locator(".block-special-promotion-banner a")
    b_count = await count(banner_nodes, "promotions")
    for i in range(b_count):
        bn = banner_nodes.nth(i)
        href = await get_attribute(bn, "href", "promotions")
        img_node = bn.locator("img")
        img_src = (
            await get_attribute(img_node, "src", "promotions")
            if await count(img_node, "promotions") > 0
            else None
        )
        img_alt = (
            await get_attribute(img_node, "alt", "promotions")
            if await count(img_node, "promotions") > 0
            else None
        )
        res["promotions"].append(
            {
                "url": (
                    href
                    if href and href.startswith("http")
                    else f"https://www.cellphones.com.vn{href or ''}"
                ),
                "img": img_src,
                "alt": img_alt,
            }
        )

    desc = await get_inner_html(
        page.locator("#cpsContentSEO"), field_name="description"
    )
    if desc:
        res["description"] = desc

    comment_rows = page.locator(".item-comment__box-cmt")
    c_count = await count(comment_rows, "comments")
    for i in range(c_count):
        section = comment_rows.nth(i)
        comment: Dict[str, Any] = {"name": None, "content": None, "reply": []}
        comment["name"] = await get_text(
            section.locator(">.box-cmt__box-info>.box-info>p.box-info__name"),
            field_name=f"comments[{i}].name",
        )
        content_nodes = section.locator(".box-cmt__box-question .content p")
        for k in range(await count(content_nodes, f"comments[{i}].content")):
            text = await get_text(
                content_nodes.nth(k), field_name=f"comments[{i}].content[{k}]"
            )
            if text:
                comment["content"] = (
                    text if not comment["content"] else comment["content"] + "\n" + text
                )
        reply_rows = section.locator(
            ">.item-comment__box-rep-comment .item-rep-comment"
        )
        r_count = await count(reply_rows, f"comments[{i}].replies")
        for j in range(r_count):
            reply_sec = reply_rows.nth(j)
            reply: Dict[str, Any] = {"name": None, "content": None}
            reply["name"] = await get_text(
                reply_sec.locator(">.box-cmt__box-info p.box-info__name"),
                field_name=f"comments[{i}].replies[{j}].name",
            )
            texts = await get_all_texts(
                reply_sec.locator(">.box-cmt__box-question .content"),
                field_name=f"comments[{i}].replies[{j}].content",
            )
            if texts:
                reply["content"] = "\n".join(texts)
            comment["reply"].append(reply)
        res["comments"].append(comment)

    return res


async def main():
    global CURRENT_ERRORS
    if not Path(ALL_LINKS_FILE).exists():
        logging.error("Links file not found: %s", ALL_LINKS_FILE)
        return

    # with open(ALL_LINKS_FILE, "r", encoding="utf-8") as f:
    #     try:
    #         all_links = json.load(f)
    #     except Exception as e:
    #         logging.exception("Failed to load links JSON: %s", e)
    #         return
    all_links = ["https://cellphones.com.vn/may-massage-da-dau-breo-scalp-mini.html", "https://cellphones.com.vn/may-massage-co-vai-gay-philips-ppm3521.html", "https://cellphones.com.vn/may-massage-cam-tay-xiaomi-massage-gun-2-gl.html", "https://cellphones.com.vn/may-massage-dau-goi-breo-x2.html", "https://cellphones.com.vn/may-massage-mat-philips-ppm3101e.html", "https://cellphones.com.vn/may-massage-mat-buheung-mk-321.html", "https://cellphones.com.vn/may-massage-co-vai-gay-philips-ppm3306n.html", "https://cellphones.com.vn/may-massage-lung-philips-ppm3111.html"]
    if not isinstance(all_links, list):
        logging.error("Links JSON must be a list")
        return

    total = len(all_links)
    logging.info("Starting crawl: %d links", total)

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
        context = await browser.new_context()
        page = await context.new_page()
        page.set_default_timeout(DEFAULT_TIMEOUT)
        await page.route("**/*", block_resources)

        processed = 0
        start_time = time.time()

        try:
            for index, url in enumerate(all_links):
                if processed > 0 and processed % RESET_PAGE_BATCH == 0:
                    try:
                        await page.close()
                    except Exception:
                        pass
                    page = await context.new_page()
                    page.set_default_timeout(DEFAULT_TIMEOUT)
                    await page.route("**/*", block_resources)

                processed += 1
                logging.info("[%d/%d] Processing: %s", processed, total, url)

                CURRENT_ERRORS = {"url": url, "errors": {}}

                nav_ok = False
                for attempt in range(1, NAV_RETRIES + 1):
                    try:
                        await page.goto(
                            url, wait_until="domcontentloaded", timeout=GOTO_TIMEOUT
                        )
                        nav_ok = True
                        break
                    except PlaywrightTimeoutError:
                        logging.warning(
                            "Timeout navigating %s (attempt %d/%d)",
                            url,
                            attempt,
                            NAV_RETRIES,
                        )
                        await asyncio.sleep(1 * attempt)
                    except Exception as e:
                        logging.warning(
                            "Error navigating %s (attempt %d/%d): %s",
                            url,
                            attempt,
                            NAV_RETRIES,
                            e,
                        )
                        await asyncio.sleep(0.5)

                if not nav_ok:
                    append_error_log_record(
                        {"url": url, "errors": {"navigation": ["nav_failed"]}}
                    )
                    continue

                try:
                    await page.wait_for_selector(".box-product-name > h1", timeout=5000)
                except Exception:
                    logging.debug("Product title not present or slow for %s", url)

                try:
                    data = await extract_item_from_page(page, url)
                    save_item(data)
                except Exception as e:
                    logging.exception("Extraction failed for %s: %s", url, e)

                if CURRENT_ERRORS and CURRENT_ERRORS.get("errors"):
                    append_error_log_record(CURRENT_ERRORS)

                await asyncio.sleep(0.05)

        except KeyboardInterrupt:
            logging.warning("KeyboardInterrupt received, shutting down gracefully...")
        finally:
            elapsed = time.time() - start_time
            logging.info(
                "Crawl finished. Processed %d links in %.1f seconds", processed, elapsed
            )
            try:
                await page.close()
            except Exception:
                pass
            try:
                await context.close()
            except Exception:
                pass
            try:
                await browser.close()
            except Exception:
                pass


if __name__ == "__main__":
    await main()


2026-02-05 20:22:01 | INFO | Starting crawl: 8 links
2026-02-05 20:22:03 | INFO | [1/8] Processing: https://cellphones.com.vn/may-massage-da-dau-breo-scalp-mini.html
2026-02-05 20:22:08 | INFO | [2/8] Processing: https://cellphones.com.vn/may-massage-co-vai-gay-philips-ppm3521.html
2026-02-05 20:22:13 | INFO | [3/8] Processing: https://cellphones.com.vn/may-massage-cam-tay-xiaomi-massage-gun-2-gl.html
2026-02-05 20:22:28 | INFO | [4/8] Processing: https://cellphones.com.vn/may-massage-dau-goi-breo-x2.html
2026-02-05 20:22:34 | INFO | [5/8] Processing: https://cellphones.com.vn/may-massage-mat-philips-ppm3101e.html
2026-02-05 20:22:46 | INFO | [6/8] Processing: https://cellphones.com.vn/may-massage-mat-buheung-mk-321.html
2026-02-05 20:23:00 | INFO | [7/8] Processing: https://cellphones.com.vn/may-massage-co-vai-gay-philips-ppm3306n.html
2026-02-05 20:23:14 | INFO | [8/8] Processing: https://cellphones.com.vn/may-massage-lung-philips-ppm3111.html
2026-02-05 20:23:27 | INFO | Crawl finis

In [9]:
#!/usr/bin/env python3
import asyncio
import json
import logging
import time
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

from logging.handlers import RotatingFileHandler

LOG_FILE = "crawler.log"

logger = logging.getLogger()
logger.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
)

# ---- Console ----
console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

# ---- File (auto rotate) ----
file_handler = RotatingFileHandler(
    LOG_FILE, maxBytes=5 * 1024 * 1024, backupCount=5, encoding="utf-8"  # 5MB
)
file_handler.setFormatter(formatter)

# ---- Attach ----
logger.handlers.clear()
logger.addHandler(console_handler)
logger.addHandler(file_handler)


from playwright.async_api import (
    async_playwright,
    TimeoutError as PlaywrightTimeoutError,
)

HEADLESS = False
SLOW_MO = 0
NAV_RETRIES = 2
GOTO_TIMEOUT = 30_000
DEFAULT_TIMEOUT = 10_000
RESET_PAGE_BATCH = 50
BLOCK_RESOURCE_TYPES = {"image", "media"}
ALL_LINKS_FILE = "all_article_links_2.json"
OUTPUT_JSONL = "article_detail.jsonl"
LOG_LEVEL = logging.INFO
CRAWL_ERROR_LOGS_FILE = "article_errors.jsonl"

logging.basicConfig(
    format="%(asctime)s %(levelname)s %(message)s",
    level=LOG_LEVEL,
    datefmt="%H:%M:%S",
)

CURRENT_ERRORS: Optional[Dict[str, Any]] = None


def append_error_log_record(record: Dict[str, Any]) -> None:
    try:
        with open(CRAWL_ERROR_LOGS_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    except Exception:
        logging.exception("Failed to write error record")


def log_error(field_name: str, source: str, err: Exception) -> None:
    global CURRENT_ERRORS
    msg = f"{source}: {type(err).__name__}: {str(err)}"
    if CURRENT_ERRORS is None:
        append_error_log_record({"url": None, "errors": {field_name: [msg]}})
        return
    errors = CURRENT_ERRORS.setdefault("errors", {})
    errors.setdefault(field_name, []).append(msg)


def save_item(item: Dict[str, Any], filename: str = OUTPUT_JSONL) -> None:
    try:
        with open(filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    except Exception:
        logging.exception("Failed to save item")


async def block_resources(route, request):
    try:
        if request.resource_type in BLOCK_RESOURCE_TYPES:
            await route.abort()
        else:
            await route.continue_()
    except Exception:
        try:
            await route.continue_()
        except Exception:
            pass


async def get_text(locator, field_name: str, strip: bool = True) -> Optional[str]:
    try:
        txt = await locator.first.text_content()
        if txt is None:
            return None
        return txt.strip() if strip else txt
    except Exception as e:
        log_error(field_name, "get_text", e)
        return None


async def get_attribute(locator, name: str, field_name: str) -> Optional[str]:
    try:
        val = await locator.first.get_attribute(name)
        return val.strip() if isinstance(val, str) else val
    except Exception as e:
        log_error(field_name, "get_attribute", e)
        return None


async def get_inner_html(locator, field_name: str) -> Optional[str]:
    try:
        html = await locator.first.inner_html()
        return html.strip() if isinstance(html, str) else html
    except Exception as e:
        log_error(field_name, "get_inner_html", e)
        return None


async def count(locator, field_name: str) -> int:
    try:
        return await locator.count()
    except Exception as e:
        log_error(field_name, "count", e)
        return 0


async def extract_article(page, url: str) -> Dict[str, Any]:
    res: Dict[str, Any] = {
        "url": url,
        "title": str,
        "updated_at": Optional[str],
        "content_html": str,
        "tags": List[str],
    }

    title = get_text(page.locator("article h1"), "title")
    if title:
        res["title"] = title

    tags = page.locator(
        ".swiper .swiper-initialized .swiper-horizontal .flex-1 .overflow-hidden .p-[2px] .swiper-backface-hidden"
    )

    for tag in tags:
        res["tags"].append(get_text(tag, "tags"))

    update_text = get_text(page.locator("span", has_text="Ngày cập nhật"), "update_at")
    if update_text:
        # tách ngày dạng dd/mm/yyyy
        m = re.search(r"(\d{1,2}/\d{1,2}/\d{4})", update_text)
        if m:
            try:
                res["updated_at"] = datetime.strptime(
                    m.group(1), "%d/%m/%Y"
                ).isoformat()

            except Exception:
                res["updated_at"] = None


async def main():
    global CURRENT_ERRORS
    url = "https://cellphones.com.vn/sforum/tag/gia-dung"
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
        context = await browser.new_context()
        page = await context.new_page()
        page.set_default_timeout(DEFAULT_TIMEOUT)
        await page.route("**/*", block_resources)

        await page.goto(url, wait_until="domcontentloaded", timeout=GOTO_TIMEOUT)
        all_article_links = set()
        filtered_article_links = set()

        while True:
            btn = page.locator('button:has-text("Xem thêm")')
            if await btn.count() == 0:
                break
            await btn.first.click()
            await page.wait_for_timeout(2000)

        article_section = page.locator('section:has-text("Bài viết về \\"Gia dụng\\"")')

        hrefs = await article_section.locator("a").evaluate_all(
            "els => els.map(e => e.getAttribute('href')).filter(Boolean)"
        )

        for link in hrefs:
            full_url = f"https://cellphones.com.vn{link}"
            all_article_links.add(full_url)
            if re.fullmatch(r"/sforum/[a-z0-9\-]+", link):
                filtered_article_links.add(full_url)

        print(
            f"all_links: {len(all_article_links)}, filtered_links: {len(filtered_article_links)}"
        )

        with open("all_article_links.json", "w", encoding="utf-8") as f:
            json.dump(list(filtered_article_links), f, ensure_ascii=False, indent=2)
                # try:
                #     await page.close()
                # except Exception:
                #     pass
                # try:
                #     await context.close()
                # except Exception:
                #     pass
                # try:
                #     await browser.close()
                # except Exception:
                #     pass


if __name__ == "__main__":
    await main()

all_links: 807, filtered_links: 763


In [24]:
import asyncio
import json
import logging
import random
import time
import re
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

from logging.handlers import RotatingFileHandler

LOG_FILE = "crawler.log"

logger = logging.getLogger()
logger.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
)

# ---- Console ----
console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

# ---- File (auto rotate) ----
file_handler = RotatingFileHandler(
    LOG_FILE, maxBytes=5 * 1024 * 1024, backupCount=5, encoding="utf-8"  # 5MB
)
file_handler.setFormatter(formatter)

# ---- Attach ----
logger.handlers.clear()
logger.addHandler(console_handler)
logger.addHandler(file_handler)


from playwright.async_api import (
    async_playwright,
    TimeoutError as PlaywrightTimeoutError,
)

HEADLESS = False
SLOW_MO = 0
NAV_RETRIES = 2
GOTO_TIMEOUT = 30_000
DEFAULT_TIMEOUT = 10_000
RESET_PAGE_BATCH = 50
BLOCK_RESOURCE_TYPES = {"image", "media"}
ALL_LINKS_FILE = "all_article_links.json"
OUTPUT_JSONL = "article_detail.jsonl"
LOG_LEVEL = logging.INFO
CRAWL_ERROR_LOGS_FILE = "article_errors.jsonl"

logging.basicConfig(
    format="%(asctime)s %(levelname)s %(message)s",
    level=LOG_LEVEL,
    datefmt="%H:%M:%S",
)

CURRENT_ERRORS: Optional[Dict[str, Any]] = None


def append_error_log_record(record: Dict[str, Any]) -> None:
    try:
        with open(CRAWL_ERROR_LOGS_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    except Exception:
        logging.exception("Failed to write error record")


def log_error(field_name: str, source: str, err: Exception) -> None:
    global CURRENT_ERRORS
    msg = f"{source}: {type(err).__name__}: {str(err)}"
    if CURRENT_ERRORS is None:
        append_error_log_record({"url": None, "errors": {field_name: [msg]}})
        return
    errors = CURRENT_ERRORS.setdefault("errors", {})
    errors.setdefault(field_name, []).append(msg)


def save_item(item: Dict[str, Any], filename: str = OUTPUT_JSONL) -> None:
    try:
        with open(filename, "a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    except Exception:
        logging.exception("Failed to save item")


async def block_resources(route, request):
    try:
        if request.resource_type in BLOCK_RESOURCE_TYPES:
            await route.abort()
        else:
            await route.continue_()
    except Exception:
        try:
            await route.continue_()
        except Exception:
            pass


async def get_text(locator, field_name: str, strip: bool = True) -> Optional[str]:
    try:
        txt = await locator.first.text_content()
        if txt is None:
            return None
        return txt.strip() if strip else txt
    except Exception as e:
        log_error(field_name, "get_text", e)
        return None


async def get_all_texts(locator, field_name: str) -> List[str]:
    try:
        texts = await locator.all_inner_texts()
        return [t.strip() for t in texts if t and t.strip()]
    except Exception as e:
        log_error(field_name, "get_all_texts", e)
        return []


async def get_attribute(locator, name: str, field_name: str) -> Optional[str]:
    try:
        val = await locator.first.get_attribute(name)
        return val.strip() if isinstance(val, str) else val
    except Exception as e:
        log_error(field_name, "get_attribute", e)
        return None


async def get_inner_html(locator, field_name: str) -> Optional[str]:
    try:
        html = await locator.first.inner_html()
        return html.strip() if isinstance(html, str) else html
    except Exception as e:
        log_error(field_name, "get_inner_html", e)
        return None


async def count(locator, field_name: str) -> int:
    try:
        return await locator.count()
    except Exception as e:
        log_error(field_name, "count", e)
        return 0


async def extract_article(page, url: str) -> Dict[str, Any]:
    res: Dict[str, Any] = {
        "url": url,
        "title": None,
        "updated_at": None,
        "cover_image": None,
        "tags": [],
        "content_html": None,
        "relevant_product": [],
    }

    title = await get_text(page.locator("article h1"), "title")
    if title:
        res["title"] = title

    update_text = await get_text(
        page.locator("span", has_text="Ngày cập nhật"), "update_at"
    )
    if update_text:
        m = re.search(r"(\d{1,2}/\d{1,2}/\d{4})", update_text)
        if m:
            try:
                res["updated_at"] = datetime.strptime(
                    m.group(1), "%d/%m/%Y"
                ).isoformat()

            except Exception:
                res["updated_at"] = None

    cover_image = await get_attribute(page.locator("div.w-full > div.w-full > img.w-full"), "src", "cover_image")
    if cover_image:
        res["cover_image"] = cover_image

    tags = await get_all_texts(
        page.locator("//span[normalize-space()='Thẻ:']/following-sibling::a"),
        "tags",
    )

    if tags:
        res["tags"] = tags
    relevant_products = page.locator(".product-info")
    rp_count = await count(relevant_products, "relevant_product")
    for i in range(rp_count):
        rp = relevant_products.nth(i)
        product: Dict[str, Any] = {
            "name": None,
            "url": None,
            "image": None,
            "base_price": None,
            "sale_price": None,
            "features": [],
        }

        name = await get_text(rp.locator("a>div>span"), f"relevant_product[{i}].name")
        if name:
            product["name"] = name
        rp_url = await get_attribute(
            rp.locator("a"),
            "href",
            f"relevant_product[{i}].url",
        )
        if rp_url:
            if rp_url.startswith("http"):
                product["url"] = rp_url
            else:
                product["url"] = f"https://cellphones.com.vn{rp_url}"
        price = rp.locator("a>div.flex")

        base_price = price.locator("span.line-through")
        bp_text = await get_text(base_price, f"relevant_product[{i}].base_price")
        if bp_text:
            product["base_price"] = bp_text
        sale_price = price.locator("span.font-[600]")
        sp_text = await get_text(sale_price, f"relevant_product[{i}].sale_price")
        features = rp.locator(".w-full.font-[500]")
        feature_texts = await get_all_texts(features, f"relevant_product[{i}].features")
        if feature_texts:
            product["features"] = feature_texts
        if sp_text:
            product["sale_price"] = sp_text
        img = await get_attribute(
            rp.locator("a > img"),
            "src",
            f"relevant_product[{i}].image",
        )
        if img:
            product["image"] = img
        res["relevant_product"].append(product)
    content_html = await get_inner_html(
        page.locator(".content-detail>.w-full").last, "content_html"
    )
    # print 2 lines of content_html
    if content_html:
        res["content_html"] = content_html
    return res


async def main():
    if not Path(ALL_LINKS_FILE).exists():
        logging.error("Links file not found: %s", ALL_LINKS_FILE)
        return

    with open(ALL_LINKS_FILE, "r", encoding="utf-8") as f:
        try:
            all_article_links = json.load(f)
        except Exception as e:
            logging.exception("Failed to load links JSON: %s", e)
            return

    if not isinstance(all_article_links, list):
        logging.error("Links JSON must be a list")
        return

    total = len(all_article_links)
    logging.info("Starting crawl: %d links", total)

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
        context = await browser.new_context()
        page = await context.new_page()
        page.set_default_timeout(DEFAULT_TIMEOUT)
        await page.route("**/*", block_resources)

        processed = 0
        start_time = time.time()

        try:
            for index, url in enumerate(all_article_links):
                if processed > 0 and processed % RESET_PAGE_BATCH == 0:
                    try:
                        await page.close()
                    except Exception:
                        pass
                    page = await context.new_page()
                    page.set_default_timeout(DEFAULT_TIMEOUT)
                    await page.route("**/*", block_resources)

                processed += 1
                logging.info("[%d/%d] Processing: %s", processed, total, url)

                CURRENT_ERRORS = {"url": url, "errors": {}}

                nav_ok = False
                for attempt in range(1, NAV_RETRIES + 1):
                    try:
                        await asyncio.sleep(random.uniform(1.5, 4.0))
                        await page.goto(
                            url, wait_until="domcontentloaded", timeout=GOTO_TIMEOUT
                        )
                        nav_ok = True
                        break
                    except PlaywrightTimeoutError:
                        logging.warning(
                            "Timeout navigating %s (attempt %d/%d)",
                            url,
                            attempt,
                            NAV_RETRIES,
                        )
                        await asyncio.sleep(1 * attempt)
                    except Exception as e:
                        logging.warning(
                            "Error navigating %s (attempt %d/%d): %s",
                            url,
                            attempt,
                            NAV_RETRIES,
                            e,
                        )
                        await asyncio.sleep(0.5)

                if not nav_ok:
                    append_error_log_record(
                        {"url": url, "errors": {"navigation": ["nav_failed"]}}
                    )
                    continue

                try:
                    data = await extract_article(page, url)
                    save_item(data)
                except Exception as e:
                    logging.exception("Extraction failed for %s: %s", url, e)

                if CURRENT_ERRORS and CURRENT_ERRORS.get("errors"):
                    append_error_log_record(CURRENT_ERRORS)

                await asyncio.sleep(0.05)

        except KeyboardInterrupt:
            logging.warning("KeyboardInterrupt received, shutting down gracefully...")
        finally:
            elapsed = time.time() - start_time
            logging.info(
                "Crawl finished. Processed %d links in %.1f seconds",
                processed,
                elapsed,
            )
            try:
                await page.close()
            except Exception:
                pass
            try:
                await context.close()
            except Exception:
                pass
            try:
                await browser.close()
            except Exception:
                pass


if __name__ == "__main__":
    await main()

2026-02-05 18:03:52 | INFO | Starting crawl: 763 links


2026-02-05 18:03:53 | ERROR | Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed\nCall log:\n  - waiting for locator("span").filter(has_text="Ngày cập nhật").first\n')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed
Call log:
  - waiting for locator("span").filter(has_text="Ngày cập nhật").first

2026-02-05 18:03:53 | INFO | [1/763] Processing: https://cellphones.com.vn/sforum/huong-dan-cach-su-dung-bep-tu-kangaroo
2026-02-05 18:03:58 | INFO | [2/763] Processing: https://cellphones.com.vn/sforum/cach-lam-goi-sua
2026-02-05 18:04:01 | INFO | [3/763] Processing: https://cellphones.com.vn/sforum/cach-lam-dua-gia-an-lien
2026-02-05 18:04:04 | INFO | [4/763] Processing: https://cellphones.com.vn/sforum/huong-dan-cach-su-dung-lo-vi-song-samsung
2026-02-05 18:04:07 | INFO | [5/763] Processing: https://cellphones.com.vn/sforum/cach-lam-sua-ngo-bang-may-lam-sua-ha